In [69]:
import pandas as pd
import numpy as np

In [2]:
instagram = pd.read_csv("instagram.csv")
tiktok = pd.read_csv("tiktok.csv")
youtube = pd.read_csv("youtube.csv")

# Filtering

## Strict Filtering

In [70]:
def filter_active_publishers(
    df,
    platform_name,
    publisher_col,
    date_col,
    min_videos_per_year
):
    """
    Keeps only publishers with at least `min_videos_per_year`
    in EVERY year from 2021 to 2026.
    """

    df = df.copy()

    # Convert date column
    df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="coerce")
    df["year"] = df[date_col].dt.year
    df = df[df["year"].between(2022, 2026)].copy()
    

    # Count videos per publisher and year
    counts = (
        df.groupby([publisher_col, "year"])
          .size()
          .reset_index(name="n_videos")
    )

    # Create complete publisher-year table
    years = sorted(df["year"].unique())

    full_index = pd.MultiIndex.from_product(
        [df[publisher_col].unique(), years],
        names=[publisher_col, "year"]
    )

    counts = (
        counts.set_index([publisher_col, "year"])
              .reindex(full_index, fill_value=0)
              .reset_index()
    )

    # Keep publishers satisfying the threshold in ALL years
    active_publishers = (
        counts.groupby(publisher_col)["n_videos"]
              .min()
              .reset_index(name="min_videos")
    )

    active_publishers = active_publishers[
        active_publishers["min_videos"] >= min_videos_per_year
    ]

    active_list = active_publishers[publisher_col].tolist()

    # Filter original dataframe
    df_active = df[df[publisher_col].isin(active_list)].copy()

    print(f"\n{platform_name}")
    print("-" * len(platform_name))
    print(f"Original publishers: {df[publisher_col].nunique()}")
    print(f"Active publishers:   {len(active_list)}")
    print(f"Original videos:     {len(df):,}")
    print(f"Remaining videos:    {len(df_active):,}")

    return df_active, active_publishers, counts

In [51]:
youtube_active, yt_publishers, yt_counts = filter_active_publishers(
    df=youtube,
    platform_name="YouTube",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=40
)


YouTube
-------
Original publishers: 130
Active publishers:   6
Original videos:     51,863
Remaining videos:    9,305


In [52]:
instagram_active, ig_publishers, ig_counts = filter_active_publishers(
    df=instagram,
    platform_name="Instagram",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=40
)


Instagram
---------
Original publishers: 161
Active publishers:   43
Original videos:     235,743
Remaining videos:    150,000


In [53]:
tiktok_active, tt_publishers, tt_counts = filter_active_publishers(
    df=tiktok,
    platform_name="TikTok",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=40
)


TikTok
------
Original publishers: 160
Active publishers:   24
Original videos:     195,672
Remaining videos:    63,929


## Less strict Filtering

In [44]:
def filter_active_publisher_years(
    df,
    platform_name,
    publisher_col,
    date_col,
    min_videos_per_year
):
    """
    Keeps only publisher-year groups with at least `min_videos_per_year` videos.
    A publisher can be retained in some years and excluded in others.
    """

    df = df.copy()

    # Convert date column
    df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="coerce")
    df["year"] = df[date_col].dt.year
    df = df[df["year"].between(2022, 2026)].copy()

    # Count videos per publisher-year
    counts = (
        df.groupby([publisher_col, "year"])
          .size()
          .reset_index(name="n_videos")
    )

    # Identify active publisher-year combinations
    active_publisher_years = counts[
        counts["n_videos"] >= min_videos_per_year
    ].copy()

    # Keep only videos belonging to active publisher-year combinations
    df_active = df.merge(
        active_publisher_years[[publisher_col, "year"]],
        on=[publisher_col, "year"],
        how="inner"
    )

    df_active["platform"] = platform_name
    active_publisher_years["platform"] = platform_name
    counts["platform"] = platform_name

    print(f"\n{platform_name}")
    print("-" * len(platform_name))
    print(f"Original videos:              {len(df):,}")
    print(f"Original publishers:          {df[publisher_col].nunique():,}")
    print(f"Videos after filtering:       {len(df_active):,}")
    print(f"Unique publishers retained:   {df_active[publisher_col].nunique():,}")
    
    print("\nPublisher-years retained by year:")
    print(
        active_publisher_years["year"]
        .value_counts()
        .sort_index()
    )

    return df_active, active_publisher_years, counts

In [64]:
youtube_active, yt_publishers, yt_counts = filter_active_publisher_years(
    df=youtube,
    platform_name="YouTube",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)


YouTube
-------
Original videos:              51,863
Original publishers:          130
Videos after filtering:       47,185
Unique publishers retained:   53

Publisher-years retained by year:
year
2022     8
2023    17
2024    24
2025    42
2026    39
Name: count, dtype: int64


Most YouTube Shorts are produced by a relatively small number of highly active channels.

In [67]:
instagram_active, ig_publishers, ig_counts = filter_active_publisher_years(
    df=instagram,
    platform_name="Instagram",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)


Instagram
---------
Original videos:              235,743
Original publishers:          161
Videos after filtering:       227,976
Unique publishers retained:   130

Publisher-years retained by year:
year
2022     30
2023     97
2024    103
2025    121
2026    103
Name: count, dtype: int64


In [68]:
tiktok_active, tt_publishers, tt_counts = filter_active_publisher_years(
    df=tiktok,
    platform_name="TikTok",
    publisher_col="publisher",
    date_col="published_at",
    min_videos_per_year=70
)


TikTok
------
Original videos:              195,672
Original publishers:          160
Videos after filtering:       189,948
Unique publishers retained:   119

Publisher-years retained by year:
year
2022    28
2023    60
2024    76
2025    94
2026    86
Name: count, dtype: int64


# Bootstrapping

In [ ]:
def bootstrap_virality_coefficients(
    df,
    platform_name,
    publisher_col,
    year_col,
    views_col="views",
    sample_size=30,
    n_bootstrap=100,
    random_state=42
):
    """
    Computes a bootstrap-based Virality Coefficient for each publisher-year.

    VC = max(views) / median(views)

    For each publisher-year:
    - draw `sample_size` videos with replacement
    - compute max / median
    - repeat `n_bootstrap` times
    - store the median VC and bootstrap uncertainty
    """

    rng = np.random.default_rng(random_state)
    results = []

    for (publisher, year), group in df.groupby([publisher_col, year_col]):

        views = group[views_col].dropna().to_numpy()

        # Remove zero or negative views to avoid division problems
        views = views[views > 0]

        if len(views) < sample_size:
            continue

        vc_values = []
        median_values = []
        max_values = []

        for _ in range(n_bootstrap):
            sample = rng.choice(
                views,
                size=sample_size,
                replace=True
            )

            sample_median = np.median(sample)
            sample_max = np.max(sample)

            if sample_median > 0:
                vc_values.append(sample_max / sample_median)
                median_values.append(sample_median)
                max_values.append(sample_max)

        if len(vc_values) == 0:
            continue

        results.append({
            "platform": platform_name,
            "publisher": publisher,
            "year": year,
            "n_videos": len(views),

            "typical_views": np.median(median_values),
            "viral_views": np.median(max_values),

            "virality_coefficient": np.median(vc_values),
            "vc_bootstrap_sd": np.std(vc_values, ddof=1),
            "vc_ci_lower": np.percentile(vc_values, 2.5),
            "vc_ci_upper": np.percentile(vc_values, 97.5)
        })

    return pd.DataFrame(results)

In [93]:
youtube_vc = bootstrap_virality_coefficients(
    df=youtube_active,
    platform_name="YouTube",
    publisher_col="publisher",
    year_col="year",
    views_col="views",
    sample_size=30,
    n_bootstrap=100
)
youtube_vc.head(5)

Results produced: 130


,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
0,YouTube,ANSA,2025,718,1280.25,29927.0,22.708867,73.510165,7.246802,292.698219
1,YouTube,ANSA,2026,759,1368.25,36949.0,26.014103,96.720492,5.297713,311.939728
2,YouTube,Adnkronos,2025,93,5227.75,208866.0,35.406313,116.623786,4.491326,346.345146
3,YouTube,Adnkronos,2026,85,4953.25,189848.0,39.195666,77.793342,6.164271,240.042679
4,YouTube,Aleteia IT,2022,129,1406.50,124041.0,83.792003,55.816429,11.477105,202.887663


In [95]:
instagram_vc = bootstrap_virality_coefficients(
    df=instagram_active,
    platform_name="Instagram",
    publisher_col="publisher",
    year_col="year",
    views_col="views",
    sample_size=30,
    n_bootstrap=100
)
instagram_vc.head(5)

Results produced: 454


,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
0,Instagram,AMICA,2023,101,75961.50,410377.0,6.058960,4.440918,2.909345,15.682331
1,Instagram,AMICA,2024,189,61093.00,626504.0,9.895736,23.931940,3.821036,78.415204
2,Instagram,AMICA,2025,243,49555.25,1015739.0,22.138925,19.011510,5.646869,65.255049
3,Instagram,AMICA,2026,103,54737.50,1086385.0,21.618939,14.827955,4.092341,52.491559
4,Instagram,Adnkronos,2023,128,8756.25,154706.0,14.738228,29.701010,2.375803,92.228733


In [96]:
tiktok_vc = bootstrap_virality_coefficients(
    df=tiktok_active,
    platform_name="TikTok",
    publisher_col="publisher",
    year_col="year",
    views_col="views",
    sample_size=30,
    n_bootstrap=100
)
tiktok_vc.head(5)

Results produced: 344


,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
0,TikTok,agi.agenziaitalia,2023,228,1228.75,170718.0,104.861534,173.645624,14.954511,624.034171
1,TikTok,agi.agenziaitalia,2024,143,1209.00,63795.0,57.058140,61.561600,27.278927,236.917506
2,TikTok,airc.it,2024,133,5116.50,392811.0,68.957864,47.306438,21.871462,186.894791
3,TikTok,airc.it,2025,148,2205.75,2646498.0,1252.389071,1013.031688,56.873279,3593.733165
4,TikTok,airc.it,2026,73,1326.00,6067726.0,4116.187195,2105.501275,102.029428,7483.128281


Combine all of them

In [99]:
vc_df = pd.concat(
    [youtube_vc, instagram_vc, tiktok_vc],
    ignore_index=True
)

vc_df.head(5)

,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
0,YouTube,ANSA,2025,718,1280.25,29927.0,22.708867,73.510165,7.246802,292.698219
1,YouTube,ANSA,2026,759,1368.25,36949.0,26.014103,96.720492,5.297713,311.939728
2,YouTube,Adnkronos,2025,93,5227.75,208866.0,35.406313,116.623786,4.491326,346.345146
3,YouTube,Adnkronos,2026,85,4953.25,189848.0,39.195666,77.793342,6.164271,240.042679
4,YouTube,Aleteia IT,2022,129,1406.50,124041.0,83.792003,55.816429,11.477105,202.887663


In [100]:
vc_df.describe()

,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
count,928.000000,928.000000,9.280000e+02,9.280000e+02,928.000000,928.000000,928.000000,928.000000
mean,2024.438578,500.080819,5.770170e+04,7.531660e+05,48.613253,78.315096,7.862864,270.987472
std,1.241367,722.003711,1.416157e+05,1.293223e+06,168.092636,208.121936,10.546759,652.492171
min,2022.000000,49.000000,1.150000e+01,6.570000e+02,1.736481,0.297697,1.380603,2.886524
25%,2023.000000,132.000000,3.253000e+03,1.096938e+05,8.053572,8.561575,2.891675,34.828087
50%,2025.000000,249.000000,1.508575e+04,3.672955e+05,19.054632,24.907951,4.673325,92.129954
75%,2025.000000,539.000000,4.933894e+04,8.462279e+05,46.158362,80.116805,8.281701,272.137790
max,2026.000000,6280.000000,1.526590e+06,1.747863e+07,4116.187195,3934.832145,143.647356,10449.830965


Which publisher-years are producing these extreme coefficients.

In [103]:
vc_df.sort_values(
    "virality_coefficient",
    ascending=False
).head(20)

,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
588,TikTok,airc.it,2026,73,1326.00,6067726.0,4116.187195,2105.501275,102.029428,7483.128281
905,TikTok,vimeo,2022,77,5556.00,11693580.0,2213.430343,1232.797666,86.079942,4708.359280
587,TikTok,airc.it,2025,148,2205.75,2646498.0,1252.389071,1013.031688,56.873279,3593.733165
801,TikTok,open_giornale,2024,94,1193.50,858565.0,718.016341,691.145027,143.647356,2261.345580
757,TikTok,lifegate,2026,96,1060.00,501140.0,485.749497,343.017820,12.261879,1015.781808
920,TikTok,wikipedia,2026,229,8012.75,3296399.0,434.178172,357.274788,20.290396,1196.973573
601,TikTok,controcopertina,2024,127,589.00,289969.0,379.487388,477.562044,47.588786,1652.933468
634,TikTok,forbes_italia,2026,112,1222.75,468381.5,363.496538,352.149638,56.800246,1133.404254
922,TikTok,wireditalia,2024,333,1166.50,361827.0,351.690155,283.526224,24.967720,983.672808
923,TikTok,wireditalia,2025,468,1411.50,542019.0,338.391674,527.391211,62.226394,1785.337273


In [ ]:
# Checking a random channel
vc_df[vc_df["publisher"] == "la Repubblica"]

,platform,publisher,year,n_videos,typical_views,viral_views,virality_coefficient,vc_bootstrap_sd,vc_ci_lower,vc_ci_upper
574,Instagram,la Repubblica,2022,259,832783.75,3308934.0,3.877049,5.005742,2.081374,21.451106
575,Instagram,la Repubblica,2023,1694,1392721.50,5991981.0,4.137691,1.806074,2.420259,8.958881
576,Instagram,la Repubblica,2024,2685,1055896.75,5204146.0,4.856678,2.200738,2.273099,11.022317
577,Instagram,la Repubblica,2025,5613,691831.00,3268062.0,4.842153,2.186241,2.994451,11.391106
578,Instagram,la Repubblica,2026,4143,500196.75,2626554.0,5.320172,2.334527,3.061666,12.158896


In [102]:
# Year evolution
vc_df.groupby(["platform","year"])["virality_coefficient"].median()

platform   year
Instagram  2022     4.173974
           2023     7.000125
           2024     8.224759
           2025    12.540783
           2026    12.669098
TikTok     2022    41.946522
           2023    54.719130
           2024    53.598593
           2025    61.623688
           2026    41.380963
YouTube    2022    53.116956
           2023    33.652102
           2024    22.468792
           2025    15.385700
           2026    12.733835
Name: virality_coefficient, dtype: float64